# 98. Macro & Market Regime 데이터 구축

## 📋 개요
종목별 데이터 수집에 앞서, 한국 시장의 전역적(Global) 환경 변수인 거시 경제 지표와 Market Regime(강세/약세/보합장) 시그널을 계산하여 별도로 저장합니다.

## ✅ 주요 기능
1. **장기 시계열 확보**: 2000년부터 현재까지의 거시 데이터 수집 (Look-back 보장)
2. **PyKRX 대체**: `FinanceDataReader`를 활용한 환율, 글로벌 지수, KOSPI 수집
3. **Regime 판단**: KOSPI 200일선 및 변동성 기반의 Bull(1), Bear(-1), Neutral(0) 상태 계산
4. **CSV Fallback**: API 수집 실패 시 `data/99_meta/*.csv` 사전 저장 데이터로 자동 대체

In [ ]:
import re
import pandas as pd
import numpy as np
import FinanceDataReader as fdr
from datetime import datetime, timedelta
from pathlib import Path

# 설정 로드 및 경로 준비 (기존 config 활용)
from src.utils.config import load_config, ProjectPaths
cfg = load_config()
paths = ProjectPaths.from_config(cfg)

# 매크로 데이터를 저장할 폴더
meta_dir = Path(cfg['paths'].get('meta_dir', 'data/99_meta'))
macro_filepath = meta_dir / "macro_regime.parquet"

print(f"📁 매크로 데이터 저장 경로: {macro_filepath}")

In [ ]:
# ==========================================
# [Helper] CSV Fallback 로더
# ==========================================
def _parse_csv_date(date_str: str) -> pd.Timestamp | None:
    """
    '2022. 7. 7 오후 3:30:00' 형태의 날짜 문자열에서 날짜(Date)만 추출.
    숫자로 시작하는 'YYYY. M. D' 부분만 파싱하고 시각은 무시.
    """
    m = re.match(r'(\d{4})\s*\.\s*(\d{1,2})\s*\.\s*(\d{1,2})', str(date_str))
    if m:
        return pd.Timestamp(f"{m.group(1)}-{m.group(2).zfill(2)}-{m.group(3).zfill(2)}")
    return None


def load_fallback_csv(indicator: str, meta_dir: Path) -> pd.Series:
    """
    data/99_meta/{indicator}.csv 를 읽어 날짜 인덱스의 Close Series 반환.

    처리 규칙:
    - Date 컬럼에서 날짜만 추출 (시각 제거)
    - 같은 날짜가 여러 행일 경우 마지막 행(장 마감 기준) 사용
    """
    csv_path = meta_dir / f"{indicator}.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Fallback CSV 없음: {csv_path}")

    df = pd.read_csv(csv_path)

    # 날짜 파싱: 'YYYY. M. D 오전/오후 H:MM:SS' → date only
    df['Date'] = df['Date'].apply(_parse_csv_date)
    df = df.dropna(subset=['Date'])

    # 같은 날짜가 여러 행인 경우 마지막 값(장 마감) 사용
    df = df.sort_values('Date').groupby('Date', sort=True).last()

    series = df['Close'].astype(float)
    series.index = pd.to_datetime(series.index)
    series.index.name = 'Date'
    series.name = indicator
    return series


print("✅ Fallback 헬퍼 함수 정의 완료")

In [ ]:
# ==========================================
# 1. 동적 날짜 설정 및 데이터 수집 (API → CSV Fallback)
# ==========================================
base_start = pd.to_datetime(cfg['data_collection']['start_date'])
base_end   = pd.to_datetime(cfg['data_collection']['end_date'])
fetch_start = (base_start - timedelta(days=365)).strftime('%Y-%m-%d')
fetch_end   = base_end.strftime('%Y-%m-%d')

print(f"📥 데이터 수집 중... ({fetch_start} ~ {fetch_end})")

macro_raw = {}
sources   = {}   # 각 지표의 수집 경로 기록 (API / CSV)

# ── KOSPI ──────────────────────────────────────────────────────────────────
try:
    macro_raw['kospi'] = fdr.DataReader('KS11', fetch_start, fetch_end)['Close']
    if macro_raw['kospi'].empty:
        raise ValueError("빈 데이터")
    sources['kospi'] = 'API'
except Exception as e:
    print(f"  ⚠️  KOSPI API 실패 ({e}) → CSV fallback")
    macro_raw['kospi'] = load_fallback_csv('kospi', meta_dir)
    sources['kospi'] = 'CSV'

# KOSPI 거래일 기준 인덱스 확정 (이후 모든 지표는 이 인덱스로 정렬)
kospi_series = macro_raw['kospi'].copy()
kospi_series.index = pd.to_datetime(kospi_series.index)
# fetch 범위 내 날짜만 유지
kospi_series = kospi_series[
    (kospi_series.index >= fetch_start) & (kospi_series.index <= fetch_end)
]
kospi_index = kospi_series.index  # KOSPI 거래일 인덱스

# ── SP500 ──────────────────────────────────────────────────────────────────
try:
    macro_raw['sp500'] = fdr.DataReader('US500', fetch_start, fetch_end)['Close']
    if macro_raw['sp500'].empty:
        raise ValueError("빈 데이터")
    sources['sp500'] = 'API'
except Exception as e:
    print(f"  ⚠️  SP500 API 실패 ({e}) → CSV fallback")
    macro_raw['sp500'] = load_fallback_csv('sp500', meta_dir)
    sources['sp500'] = 'CSV'

# ── USD/KRW ────────────────────────────────────────────────────────────────
try:
    macro_raw['usd_krw'] = fdr.DataReader('USD/KRW', fetch_start, fetch_end)['Close']
    if macro_raw['usd_krw'].empty:
        raise ValueError("빈 데이터")
    sources['usd_krw'] = 'API'
except Exception as e:
    print(f"  ⚠️  USD/KRW API 실패 ({e}) → CSV fallback")
    macro_raw['usd_krw'] = load_fallback_csv('usd_krw', meta_dir)
    sources['usd_krw'] = 'CSV'

# ── VIX ───────────────────────────────────────────────────────────────────
try:
    _vix = fdr.DataReader('FRED:VIXCLS', fetch_start, fetch_end)['VIXCLS']
    if _vix.empty:
        raise ValueError("빈 데이터")
    macro_raw['vix'] = _vix
    sources['vix'] = 'API'
except Exception as e:
    print(f"  ⚠️  VIX API 실패 ({e}) → CSV fallback")
    try:
        macro_raw['vix'] = load_fallback_csv('vix', meta_dir)
        sources['vix'] = 'CSV'
    except FileNotFoundError:
        print("  ⚠️  VIX CSV도 없음 → 빈 Series 사용")
        macro_raw['vix'] = pd.Series(dtype=float)
        sources['vix'] = 'EMPTY'

# ==========================================
# 2. KOSPI 거래일 기준으로 인덱스 정렬
#
# 규칙:
#   - KOSPI 거래일에 없는 타 지표 값 → ffill(직전 거래일 값으로 대체)
#   - KOSPI 휴장일의 타 지표 값 → 보간 목적 제외 시 무시(reindex로 자동 제거)
# ==========================================
aligned = {'kospi': kospi_series}

for key in ['sp500', 'usd_krw', 'vix']:
    s = macro_raw[key].copy()
    s.index = pd.to_datetime(s.index)
    if s.empty:
        aligned[key] = pd.Series(np.nan, index=kospi_index, name=key)
        continue

    # kospi_index + 원본 날짜를 합친 임시 인덱스로 reindex 후 ffill → KOSPI 날짜만 추출
    # (이렇게 하면 KOSPI 거래일에 해당 지표 데이터가 없어도 직전 값으로 채워짐)
    combined_index = s.index.union(kospi_index).sort_values()
    s_expanded = s.reindex(combined_index).ffill()
    aligned[key] = s_expanded.reindex(kospi_index)

df_macro = pd.DataFrame(aligned)
df_macro.index.name = 'Date'

print(f"\n✅ 수집 완료: {len(df_macro):,} 거래일")
print("   수집 경로:")
for k, v in sources.items():
    print(f"     {k:<10}: {v}")
print(df_macro.tail(3))

In [ ]:
# ==========================================
# 2. Market Regime 및 파생 피처 계산
# ==========================================
print("⚙️ Market Regime 및 캘린더 피처 계산 중...")

# 1. KOSPI 기술적 지표 계산
df_macro['kospi_ma200'] = df_macro['kospi'].rolling(window=200).mean()
df_macro['kospi_vol20'] = df_macro['kospi'].pct_change().rolling(window=20).std()

# 동적 임계값: 최근 1년(250거래일) 변동성 중위수
vol_median = df_macro['kospi_vol20'].rolling(window=250).median()

# 2. Regime 판단 로직
#   - Bull (1) : 주가가 200일선 위
#   - Bear (-1): 주가가 200일선 아래 & 단기 변동성이 장기 중위수보다 큼 (투매 장세)
#   - Neutral (0): 그 외 횡보장
conditions = [
    (df_macro['kospi'] > df_macro['kospi_ma200']),
    (df_macro['kospi'] < df_macro['kospi_ma200']) & (df_macro['kospi_vol20'] > vol_median)
]
choices = [1, -1]
df_macro['market_regime'] = np.select(conditions, choices, default=0)

# 3. 미국 시장 수익률
# 한국 시초가에 영향을 주는 전일(혹은 직전 거래일) 미국 시장 수익률
df_macro['us_return_1d'] = df_macro['sp500'].pct_change().shift(1)

# 4. 결측치 제거 (초기 250일분 이동평균 계산 구간 제거)
df_macro = df_macro.dropna()

print("✅ 계산 완료. Regime 분포:")
print(df_macro['market_regime'].value_counts(normalize=True).map('{:.1%}'.format))

In [ ]:
# ==========================================
# 3. 데이터 저장
# ==========================================
# 인덱스(Date)를 컬럼으로 리셋하여 병합하기 편하게 만듭니다.
df_macro_final = df_macro.reset_index()
if 'Date' in df_macro_final.columns:
    df_macro_final = df_macro_final.rename(columns={'Date': 'date'})
elif 'index' in df_macro_final.columns:
    df_macro_final = df_macro_final.rename(columns={'index': 'date'})

# 필요한 핵심 피처만 선택 (KOSPI MA 등은 레짐 계산용이므로 제외 가능하나 분석용으로 남겨둠)
final_cols = ['date', 'kospi', 'usd_krw', 'vix', 'us_return_1d', 'market_regime']
df_save = df_macro_final[final_cols].copy()

# Parquet 포맷으로 저장
meta_dir.mkdir(parents=True, exist_ok=True)
df_save.to_parquet(macro_filepath, index=False)
# CSV 포맷으로 저장
csv_filepath = meta_dir / "macro_regime.csv"
df_save.to_csv(csv_filepath, index=False)

print(f"💾 매크로 데이터 저장 완료! -> {macro_filepath}")
print(f"   - 총 데이터 수: {len(df_save):,}일")
display(df_save.head())